<a href="https://colab.research.google.com/github/Suwetha2210/AEGIS_2207/blob/main/AEGIS_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pyngrok nest-asyncio cryptography

In [2]:
# ============================================
# AEGIS — MODULE 4: SCOPE ENFORCEMENT
# Capability Token System
# "Agent ku only what it needs"
# ============================================

import json
import datetime
import hashlib
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.hazmat.primitives import serialization
from cryptography.exceptions import InvalidSignature

# ============================================
# SETUP — Keys (same as before)
# ============================================

private_key = Ed25519PrivateKey.generate()
public_key = private_key.public_key()

private_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
).decode()

public_bytes = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

print("✅ Keys ready\n")

# ============================================
# PART A: CAPABILITY TOKEN
# ============================================

class CapabilityToken:
    def __init__(self, private_key_pem):
        self.private_key = serialization.load_pem_private_key(
            private_key_pem.encode(),
            password=None
        )

    def create_token(self,
                     owner_name,
                     agent_name,
                     allowed_actions,
                     allowed_domains,
                     max_amount,
                     valid_hours=1):
        """
        Scoped token create pannuvom
        Owner sign pannuvom — agent carry pannuvom
        """

        # Token content — scope define pannuvom
        token_data = {
            "owner": owner_name,
            "agent": agent_name,
            "allowed_actions": allowed_actions,
            "allowed_domains": allowed_domains,
            "max_amount": max_amount,
            "currency": "INR",
            "created_at": datetime.datetime.utcnow().isoformat(),
            "expires_at": (
                datetime.datetime.utcnow() +
                datetime.timedelta(hours=valid_hours)
            ).isoformat(),
            "token_id": hashlib.sha256(
                f"{owner_name}{agent_name}{datetime.datetime.utcnow()}".encode()
            ).hexdigest()[:16]
        }

        # Sign the token — owner's private key
        token_bytes = json.dumps(token_data, sort_keys=True).encode()
        signature = self.private_key.sign(token_bytes)

        return {
            "token_data": token_data,
            "signature": signature.hex()
        }


# ============================================
# PART B: SCOPE ENFORCER
# ============================================

class ScopeEnforcer:
    def __init__(self, public_key_pem):
        self.public_key = serialization.load_pem_public_key(
            public_key_pem.encode()
        )
        self.violation_log = []

    def verify_token(self, token_package):
        """
        Token valid ah nu check pannuvom
        3 checks:
        1. Signature valid ah?
        2. Expired ah?
        3. Token tampered ah?
        """
        token_data = token_package["token_data"]
        signature = bytes.fromhex(token_package["signature"])
        token_bytes = json.dumps(token_data, sort_keys=True).encode()

        # Check 1: Signature verify
        try:
            self.public_key.verify(signature, token_bytes)
        except InvalidSignature:
            return False, "❌ Token signature invalid — forged or tampered"

        # Check 2: Expiry check
        expires_at = datetime.datetime.fromisoformat(
            token_data["expires_at"]
        )
        if datetime.datetime.utcnow() > expires_at:
            return False, "❌ Token expired"

        return True, "✅ Token valid"

    def check_action(self, token_package, action, domain, amount):
        """
        Agent oru action try pannuvom pothu
        Scope check pannuvom — allowed ah?

        4 checks:
        1. Token valid ah?
        2. Action allowed ah?
        3. Domain allowed ah?
        4. Amount within limit ah?
        """
        print(f"\n--- Checking: {action} on {domain} for ₹{amount} ---")

        # Check 1: Token valid ah?
        is_valid, message = self.verify_token(token_package)
        if not is_valid:
            self._log_violation(action, domain, amount, message)
            print(message)
            return False

        token_data = token_package["token_data"]

        # Check 2: Action allowed ah?
        if action not in token_data["allowed_actions"]:
            reason = f"❌ Action '{action}' not in scope {token_data['allowed_actions']}"
            self._log_violation(action, domain, amount, reason)
            print(reason)
            return False

        # Check 3: Domain allowed ah?
        domain_allowed = any(
            allowed in domain
            for allowed in token_data["allowed_domains"]
        )
        if not domain_allowed:
            reason = f"❌ Domain '{domain}' not in scope {token_data['allowed_domains']}"
            self._log_violation(action, domain, amount, reason)
            print(reason)
            return False

        # Check 4: Amount within limit ah?
        if amount > token_data["max_amount"]:
            reason = f"❌ Amount ₹{amount} exceeds limit ₹{token_data['max_amount']}"
            self._log_violation(action, domain, amount, reason)
            print(reason)
            return False

        print(f"✅ Action ALLOWED — all scope checks passed")
        return True

    def _log_violation(self, action, domain, amount, reason):
        """
        Every violation log pannuvom
        AEGIS audit trail ku connect aaguthu
        """
        self.violation_log.append({
            "attempted_action": action,
            "attempted_domain": domain,
            "attempted_amount": amount,
            "reason": reason,
            "timestamp": datetime.datetime.utcnow().isoformat()
        })

    def get_violation_report(self):
        return self.violation_log


# ============================================
# PART C: FULL DEMO
# ============================================

print("=" * 55)
print("AEGIS MODULE 4 — SCOPE ENFORCEMENT DEMO")
print("=" * 55)

# ---- STEP 1: Owner creates scoped token for agent ----
print("\n[STEP 1] Suwetha creates capability token for her agent\n")

token_creator = CapabilityToken(private_bytes)

suwetha_token = token_creator.create_token(
    owner_name="Suwetha",
    agent_name="Suwetha_Agent_v1",
    allowed_actions=["PAY_BILL", "VIEW_BALANCE", "DOWNLOAD_RECEIPT"],
    allowed_domains=["tneb.in", "bescom.org"],
    max_amount=1000,
    valid_hours=1
)

print(f"Token created for: {suwetha_token['token_data']['agent']}")
print(f"Owner: {suwetha_token['token_data']['owner']}")
print(f"Allowed actions: {suwetha_token['token_data']['allowed_actions']}")
print(f"Allowed domains: {suwetha_token['token_data']['allowed_domains']}")
print(f"Max amount: ₹{suwetha_token['token_data']['max_amount']}")
print(f"Valid until: {suwetha_token['token_data']['expires_at']}")
print(f"Token ID: {suwetha_token['token_data']['token_id']}")

# ---- STEP 2: Enforcer setup ----
print("\n[STEP 2] AEGIS Scope Enforcer ready\n")
enforcer = ScopeEnforcer(public_bytes)

# ---- STEP 3: Legitimate actions ----
print("=" * 55)
print("LEGITIMATE ACTIONS — Should all pass")
print("=" * 55)

enforcer.check_action(
    suwetha_token,
    action="PAY_BILL",
    domain="tneb.in",
    amount=800
)

enforcer.check_action(
    suwetha_token,
    action="VIEW_BALANCE",
    domain="tneb.in",
    amount=0
)

enforcer.check_action(
    suwetha_token,
    action="DOWNLOAD_RECEIPT",
    domain="bescom.org",
    amount=0
)

# ---- STEP 4: Attacker / out of scope actions ----
print("\n" + "=" * 55)
print("OUT OF SCOPE ACTIONS — Should all block")
print("=" * 55)

# Attempt 1: Wrong action
enforcer.check_action(
    suwetha_token,
    action="SEND_EMAIL",        # Not in allowed actions
    domain="gmail.com",
    amount=0
)

# Attempt 2: Wrong domain
enforcer.check_action(
    suwetha_token,
    action="PAY_BILL",
    domain="attacker.com",      # Not in allowed domains
    amount=800
)

# Attempt 3: Over limit
enforcer.check_action(
    suwetha_token,
    action="PAY_BILL",
    domain="tneb.in",
    amount=50000                # Over ₹1000 limit
)

# Attempt 4: Prompt injection trying to access bank
enforcer.check_action(
    suwetha_token,
    action="TRANSFER_FUNDS",    # Not in scope
    domain="sbi.co.in",         # Not in scope
    amount=80000                # Way over limit
)

# ---- STEP 5: Violation report ----
print("\n" + "=" * 55)
print("AEGIS VIOLATION REPORT")
print("=" * 55)

violations = enforcer.get_violation_report()
print(f"\nTotal violations caught: {len(violations)}\n")

for i, v in enumerate(violations, 1):
    print(f"Violation {i}:")
    print(f"  Attempted: {v['attempted_action']} on {v['attempted_domain']}")
    print(f"  Amount: ₹{v['attempted_amount']}")
    print(f"  Reason: {v['reason']}")
    print(f"  Time: {v['timestamp']}")
    print()

# ---- STEP 6: Forged token attempt ----
print("=" * 55)
print("BONUS: ATTACKER TRIES FORGED TOKEN")
print("=" * 55)

# Attacker creates their own token
attacker_private = Ed25519PrivateKey.generate()
attacker_private_pem = attacker_private.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
).decode()

attacker_token_creator = CapabilityToken(attacker_private_pem)

fake_token = attacker_token_creator.create_token(
    owner_name="Suwetha",       # Pretending to be Suwetha
    agent_name="HACKED_AGENT",
    allowed_actions=["TRANSFER_FUNDS", "DELETE_RECORDS"],
    allowed_domains=["sbi.co.in", "attacker.com"],
    max_amount=999999,
    valid_hours=999
)

print("\nAttacker created fake token pretending to be Suwetha...")
print("Trying to use it...\n")

enforcer.check_action(
    fake_token,
    action="TRANSFER_FUNDS",
    domain="sbi.co.in",
    amount=999999
)

print("\n🛡️ Forged token rejected — attacker's signature doesn't match Suwetha's public key")

# ---- FINAL SUMMARY ----
print("\n" + "=" * 55)
print("AEGIS MODULE 4 COMPLETE")
print("=" * 55)
print("""
Legitimate actions:  3/3 ✅ ALLOWED
Out of scope:        4/4 ❌ BLOCKED
Forged token:        1/1 ❌ BLOCKED

AEGIS Scope Enforcement: WORKING
""")
print("🛡️ Agent accessed ONLY what Suwetha permitted. Nothing more.")

✅ Keys ready

AEGIS MODULE 4 — SCOPE ENFORCEMENT DEMO

[STEP 1] Suwetha creates capability token for her agent

Token created for: Suwetha_Agent_v1
Owner: Suwetha
Allowed actions: ['PAY_BILL', 'VIEW_BALANCE', 'DOWNLOAD_RECEIPT']
Allowed domains: ['tneb.in', 'bescom.org']
Max amount: ₹1000
Valid until: 2026-08-21T09:34:26.765488
Token ID: fd226731d06bf5ba

[STEP 2] AEGIS Scope Enforcer ready

LEGITIMATE ACTIONS — Should all pass

--- Checking: PAY_BILL on tneb.in for ₹800 ---
✅ Action ALLOWED — all scope checks passed

--- Checking: VIEW_BALANCE on tneb.in for ₹0 ---
✅ Action ALLOWED — all scope checks passed

--- Checking: DOWNLOAD_RECEIPT on bescom.org for ₹0 ---
✅ Action ALLOWED — all scope checks passed

OUT OF SCOPE ACTIONS — Should all block

--- Checking: SEND_EMAIL on gmail.com for ₹0 ---
❌ Action 'SEND_EMAIL' not in scope ['PAY_BILL', 'VIEW_BALANCE', 'DOWNLOAD_RECEIPT']

--- Checking: PAY_BILL on attacker.com for ₹800 ---
❌ Domain 'attacker.com' not in scope ['tneb.in', 'bescom

/tmp/ipykernel_831/1785461553.py:65: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.datetime.utcnow().isoformat(),
/tmp/ipykernel_831/1785461553.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.datetime.utcnow() +
/tmp/ipykernel_831/1785461553.py:71: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"{owner_name}{agent_name}{datetime.datetime.utcnow()}".encode()
/tmp/ipykernel_831/1785461553.py:118: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal i